In [ ]:
# Base folder in S3 where the 4 CSV files are stored
DATA_DIR = "s3://olist-spark-project/olist-spark-project"
# Folder in S3 where we'll save all our results
OUT_DIR = "s3://olist-spark-project/output"

#header=True uses row 1 as column names, inferSchema=True auto-detects data types
orders = spark.read.csv(f"{DATA_DIR}/olist_orders_dataset.csv", header=True, inferSchema=True)
items = spark.read.csv(f"{DATA_DIR}/olist_order_items_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA_DIR}/olist_products_dataset.csv", header=True, inferSchema=True)
reviews = spark.read.csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv",header=True,inferSchema=True,multiLine=True,escape='"'   )
print(orders.count(), items.count(), products.count(), reviews.count())

99441 112650 32951 104162


In [17]:
import time, json
from pyspark.sql import functions as F # F = Spark's toolbox of functions (avg, count, filter conditions, etc.)
from pyspark.sql.functions import broadcast # broadcast = tells Spark to copy a small table to every worker instead of shuffling it 

def clean_and_join(orders, items, products, reviews, use_broadcast=False):
    delivered = orders.filter(F.col("order_status") == "delivered")
    delivered = delivered.withColumn("delivery_delay_days",
        (F.col("order_delivered_customer_date").cast("long") -
         F.col("order_estimated_delivery_date").cast("long")) / 86400.0)
    df = delivered.join(items, on="order_id", how="inner")
    if use_broadcast:
        df = df.join(broadcast(products), on="product_id", how="inner")
    else:
        df = df.join(products, on="product_id", how="inner")
    df = df.join(reviews, on="order_id", how="inner")
    df = df.withColumn("is_bad_review", (F.col("review_score") <= 3).cast("int"))
    df = df.na.drop(subset=["delivery_delay_days", "product_category_name", "is_bad_review"])
    return df.select("order_id", "product_category_name", "delivery_delay_days",
                      "is_bad_review", "review_score", "price", "freight_value")

def aggregate(df):
    return (df.groupBy("product_category_name")
        .agg(F.avg("delivery_delay_days").alias("avg_delivery_delay_days"),
             F.avg("is_bad_review").alias("bad_review_rate"),
             F.count("order_id").alias("n_orders"))
        .orderBy(F.desc("bad_review_rate")))

print("Functions ready.")

Functions ready.


In [18]:
results = {"stress_test": [], "ml": {}, "broadcast_join": {}}

t0 = time.perf_counter()
base_df = clean_and_join(orders, items, products, reviews).cache()
n_rows = base_df.count()
join_time = time.perf_counter() - t0
print(f"join+clean time: {join_time:.3f}s, rows: {n_rows:,}")

results["base_join_time_sec"] = join_time
results["base_rows"] = n_rows

join+clean time: 3.587s, rows: 108,472


In [ ]:
# Run our aggregation function on the cleaned data
agg_result = aggregate(base_df)
# Display the top 20 rows in full (truncate=False means don't cut off long category names)
agg_result.show(agg_result.count(), truncate=False )
print(f"Rows: {agg_result.count()}, Columns: {len(agg_result.columns)}")

+----------------------------------------------+-----------------------+-------------------+--------+
|product_category_name                         |avg_delivery_delay_days|bad_review_rate    |n_orders|
+----------------------------------------------+-----------------------+-------------------+--------+
|fraldas_higiene                               |-10.658655843343345    |0.5135135135135135 |37      |
|portateis_cozinha_e_preparadores_de_alimentos |-8.765803571428572     |0.5                |14      |
|seguros_e_servicos                            |-16.27826388888889     |0.5                |2       |
|moveis_escritorio                             |-11.132146135205211    |0.39663461538461536|1664    |
|casa_conforto_2                               |-7.309356995884774     |0.37037037037037035|27      |
|artigos_de_festas                             |-14.34000385802469     |0.3333333333333333 |42      |
|casa_conforto                                 |-9.162442641042208     |0.32558139

In [20]:
agg_result.write.mode("overwrite").option("header", True).csv(f"{OUT_DIR}/spark_aggregation_by_category")
print("Saved.")

Saved.


In [21]:
# this is broadcast join vs default shuffle join
t0 = time.perf_counter()
clean_and_join(orders, items, products, reviews, use_broadcast=False).count()
shuffle_time = time.perf_counter() - t0

t0 = time.perf_counter()
clean_and_join(orders, items, products, reviews, use_broadcast=True).count()
broadcast_time = time.perf_counter() - t0

print(f"shuffle join: {shuffle_time:.3f}s | broadcast join: {broadcast_time:.3f}s")
results["broadcast_join"] = {"shuffle_join_time_sec": shuffle_time, "broadcast_join_time_sec": broadcast_time}

shuffle join: 0.300s | broadcast join: 0.237s


In [ ]:
for factor in [1, 5, 10, 20]:
    rep_df = base_df
    for _ in range(factor - 1):
        rep_df = rep_df.unionAll(base_df)
    rep_df = rep_df.cache()
    rows = rep_df.count()

    t0 = time.perf_counter()
    agg = aggregate(rep_df)
    agg.count()
    elapsed = time.perf_counter() - t0

    print(f"factor={factor:>2} rows={rows:>10,} time={elapsed:.4f}s")
    results["stress_test"].append({"factor": factor, "n_rows": rows, "time_sec": elapsed, "status": "ok"})
    rep_df.unpersist()

73
factor= 1 rows=   108,472 time=0.2247s
DataFrame[order_id: string, product_category_name: string, delivery_delay_days: double, is_bad_review: int, review_score: string, price: double, freight_value: double]
73
factor= 5 rows=   542,360 time=0.4342s
DataFrame[order_id: string, product_category_name: string, delivery_delay_days: double, is_bad_review: int, review_score: string, price: double, freight_value: double]
73
factor=10 rows= 1,084,720 time=0.5301s
DataFrame[order_id: string, product_category_name: string, delivery_delay_days: double, is_bad_review: int, review_score: string, price: double, freight_value: double]
73
factor=20 rows= 2,169,440 time=0.8183s
DataFrame[order_id: string, product_category_name: string, delivery_delay_days: double, is_bad_review: int, review_score: string, price: double, freight_value: double]


In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Set up the indexer: converts category names (text) into numeric IDs
# handleInvalid="keep" avoids crashing on unseen categories
indexer = StringIndexer(inputCol="product_category_name", outputCol="category_idx", handleInvalid="keep")
# Set up the assembler: combines our two input columns (delay + category number) into one "features" column the model needs
assembler = VectorAssembler(inputCols=["delivery_delay_days", "category_idx"], outputCol="features")
train_df, test_df = base_df.randomSplit([0.8, 0.2], seed=42)

# Train the category encoder using only training data
# Then convert the text category in training data into numbers
# Finally, combine delay and category into the "features" column
vec_train = assembler.transform(indexer.fit(train_df).transform(train_df))
lr = LogisticRegression(featuresCol="features", labelCol="is_bad_review", maxIter=50)

t0 = time.perf_counter()
model = lr.fit(vec_train)
train_time = time.perf_counter() - t0

# Convert the test dataset categories into numbers
# Then combine test features into the required "features" column
vec_test = assembler.transform(indexer.fit(test_df).transform(test_df))
preds = model.transform(vec_test)
auc = BinaryClassificationEvaluator(labelCol="is_bad_review", metricName="areaUnderROC").evaluate(preds)

print(f"train_time={train_time:.3f}s areaUnderROC={auc:.4f}")
results["ml"] = {"train_time_sec": train_time, "areaUnderROC": auc}

train_time=6.283s areaUnderROC=0.5997


In [ ]:
with open("/tmp/spark_results_emr.json", "w") as f:
    json.dump(results, f, indent=2)

import subprocess
subprocess.run(["aws", "s3", "cp", "/tmp/spark_results_emr.json", f"{OUT_DIR}/spark_results_emr.json"])
print("All results saved to:", OUT_DIR)